<a href="https://colab.research.google.com/github/moosjiny/connectai/blob/main/Roops_System_Infer_Server.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install flask pyngrok transformers torch accelerate -q

In [2]:
%%writefile colab_infer_server.py
import json, os, threading, time
import torch
from flask import Flask, jsonify, request
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = os.getenv("COLAB_MODEL", "Qwen/Qwen2.5-0.5B-Instruct")
API_KEY    = os.getenv("COLAB_API_KEY", "")
NTFY_TOKEN = os.getenv("NTFY_TOKEN", "")
NTFY_URL   = "https://ntfy.hyperbook.com/roops-eros"
PORT       = int(os.getenv("PORT", "5000"))

print(f"모델 로드 중: {MODEL_NAME}")
_tok   = AutoTokenizer.from_pretrained(MODEL_NAME)
_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float16)
_model = _model.to("cuda" if torch.cuda.is_available() else "cpu")
_model.eval()
print(f"모델 로드 완료 (device: {next(_model.parameters()).device})")

def infer(state, instructions, options):
    보기 = " / ".join(f"{k}: {v}" for k, v in options.items())
    prompt = (
        f"상황: {state}\n질문: {instructions}\n보기: {보기}\n"
        f"가장 알맞은 보기의 알파벳 하나만 고르시오.\n답:"
    )
    ids = _tok(prompt, return_tensors="pt").input_ids.to(_model.device)
    with torch.no_grad():
        logits = _model(ids).logits[0, -1]
    option_ids = {k: _tok.encode(k, add_special_tokens=False)[0] for k in options}
    raw   = torch.stack([logits[i] for i in option_ids.values()])
    probs = torch.softmax(raw, 0)
    return {k: round(float(v), 4) for k, v in zip(options, probs)}

app = Flask(__name__)

def _auth_ok():
    if not API_KEY: return True
    return request.headers.get("X-Api-Key", "") == API_KEY

@app.route("/health")
def health():
    return jsonify({"status": "ok", "model": MODEL_NAME, "device": str(next(_model.parameters()).device)})

@app.route("/infer", methods=["POST"])
def infer_endpoint():
    if not _auth_ok():
        return jsonify({"error": "unauthorized"}), 401
    body = request.get_json(force=True)
    try:
        return jsonify(infer(body["state"], body["instructions"], body["options"]))
    except KeyError as e:
        return jsonify({"error": f"필드 누락: {e}"}), 400

def _notify_url(public_url):
    if not NTFY_TOKEN:
        print(f"[ntfy] NTFY_TOKEN 없음 — URL: {public_url}")
        return
    import urllib.request
    msg = f"[Colab] System1 infer URL 갱신\n{public_url}\n\n[발신: colab (Hermes-GPU-Bridge) · ntfy.hyperbook.com]"
    req = urllib.request.Request(
        NTFY_URL, data=msg.encode(),
        headers={"Authorization": f"Bearer {NTFY_TOKEN}", "Title": "[Colab] System1 infer URL updated"},
        method="POST",
    )
    with urllib.request.urlopen(req) as r:
        print(f"[ntfy] 전송 완료 ({r.status}): {public_url}")

def _start_cloudflare():
    import re, subprocess, shutil
    try:
        if not shutil.which("cloudflared"):
            print("[cloudflare] cloudflared 다운로드 중...")
            subprocess.run(["wget", "-q",
                "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
                "-O", "/usr/local/bin/cloudflared"], check=True)
            subprocess.run(["chmod", "+x", "/usr/local/bin/cloudflared"], check=True)
        proc = subprocess.Popen(
            ["cloudflared", "tunnel", "--url", f"http://localhost:{PORT}"],
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
        )
        deadline = time.time() + 30
        for line in proc.stdout:
            print(f"[cloudflare] {line.rstrip()}")
            m = re.search(r"https://[a-z0-9\-]+\.trycloudflare\.com", line)
            if m:
                url = m.group(0)
                print(f"[cloudflare] 공개 URL: {url}")
                threading.Thread(target=_notify_url, args=(url,), daemon=True).start()
                return url
            if time.time() > deadline:
                break
        print("[cloudflare] URL 획득 실패")
    except Exception as e:
        print(f"[cloudflare] 실패: {e}")

if __name__ == "__main__":
    flask_thread = threading.Thread(
        target=lambda: app.run(host="0.0.0.0", port=PORT, threaded=True), daemon=True)
    flask_thread.start()
    time.sleep(1)
    _start_cloudflare()
    flask_thread.join()

Overwriting colab_infer_server.py


In [ ]:
import os
os.environ["COLAB_API_KEY"] = "system1-colab-key"
os.environ["NTFY_TOKEN"] = "tk_oe2sjvk2q34iafpewlmwcx5q5zc16"
os.environ["PORT"] = "5000"
!python colab_infer_server.py

모델 로드 중: Qwen/Qwen2.5-0.5B-Instruct
config.json: 100% 659/659 [00:00<00:00, 3.13MB/s]
tokenizer_config.json: 100% 7.30k/7.30k [00:00<00:00, 17.1MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 66.0MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 124MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 152MB/s]

model.safetensors: downloading bytes:   7% 64.5M/988M [00:00<00:07, 128MB/s, 3.00MB/s  ] 
model.safetensors: downloading bytes:  13% 125M/988M [00:01<00:05, 160MB/s, 10.9MB/s  ] 
model.safetensors: downloading bytes:  43% 426M/988M [00:02<00:01, 286MB/s, 37.2MB/s  ]
model.safetensors: downloading bytes:  47% 462M/988M [00:02<00:02, 243MB/s, 39.1MB/s  ]
model.safetensors: downloading bytes:  55% 542M/988M [00:02<00:01, 258MB/s, 44.5MB/s  ]
model.safetensors: downloading bytes:  75% 739M/988M [00:03<00:00, 259MB/s, 60.9MB/s  ]
model.safetensors: downloading bytes:  82% 809M/988M [00:03<00:00, 208MB/s, 65.4MB/s  ]
model.safetensors: downloading bytes:  86% 847M/988M [00:04<00:00, 19